# Tính năng 3 – Trang **Thị trường việc làm**

Trang có ba chế độ: **Nhu cầu tuyển dụng** (lọc theo nguồn, nhóm nghề, địa điểm, hình thức), **Mức lương** và **Kinh nghiệm**.

Khác với trang Tổng quan (1.735 tin salary-v2), trang này chạy trên **quy trình kiểm duyệt nghiêm ngặt ban đầu** của dự án (ADR 0005, 0007): chỉ những tin vượt qua cổng kiểm tra bằng chứng mới được dùng, và một nhãn lương chỉ được tính khi đã được duyệt tay. Vì vậy số tin ở đây nhỏ hơn nhiều – đây là cố ý, để thấy rõ khác biệt giữa "ít nhưng chắc chắn" và "nhiều nhưng có giả định".

Mã nguồn: `pages/market.py`, `src/career_market/service.py`, `src/career_market/dashboard.py`.

In [ ]:
# Thiết lập: dùng đúng thư viện của ứng dụng (job_market/src), dữ liệu và mô hình đã lưu.
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "career_market").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from career_market import salary_charts as charts  # thư viện biểu đồ chọn backend "Agg" (vẽ cho web)…
%matplotlib inline
# …nên bật lại chế độ inline để biểu đồ hiện trong notebook.
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [ ]:
from career_market.service import MarketService

service = MarketService()
report = service.prepare()
pd.Series(report["summary"], name="Số dòng").to_frame()

Phễu ở trên: trong 3.606 dòng thô của hai nguồn (TopCV 2025–2026 và CareerBuilder 2023), phần lớn bị loại vì **ngoài phạm vi** (không phải nghề CNTT hoặc không đủ bằng chứng), một phần bị đưa sang **duyệt vai trò**; chỉ 320 tin vào phân tích (EDA) và chỉ vài tin có nhãn lương được duyệt.

**Vì sao giữ quy trình nghiêm ngặt này cho trang Thị trường?** Đây là phiên bản "đoán ít nhưng đúng" – mỗi con số đều truy được về tin gốc. Bài học của dự án là cách này quá ít để dạy mô hình (14 nhãn), nhưng vẫn hữu ích để mô tả nhu cầu kỹ năng.

## 1. Nhu cầu kỹ năng theo bộ lọc
Quy ước: bộ lọc rỗng = không lọc; một tin chỉ tính một lần; tỉ lệ % tính trên số tin của phần đã lọc.

In [ ]:
def demand(role=None, locations=(), work_mode=None, sources=("jobs-2026",)):
    result = service.skill_demand(report["eda"], role=role, locations=list(locations), work_mode=work_mode, source_ids=list(sources))
    return result, pd.DataFrame(result["table"])

cases = {
    "Toàn bộ (TopCV 2025–2026)": dict(),
    "Backend": dict(role="Backend"),
    "Data/ML": dict(role="Data/ML"),
    "Remote": dict(work_mode="Remote"),
    "Cả hai nguồn": dict(sources=("jobs-2026", "jobs-2023")),
}
rows = []
for name, kwargs in cases.items():
    result, table = demand(**kwargs)
    rows.append({"Bộ lọc": name, "Số tin": result["posting_count"], "Tin có kỹ năng": result["with_skill_count"],
                 "Top 3 kỹ năng": ", ".join(table["skill"].head(3)) if len(table) else "—"})
pd.DataFrame(rows)

In [ ]:
from IPython.display import Image, display

result, table = demand()
print(result["scope"])
display(Image(result["png"]))
table.head(10)

⚠️ Nguồn TopCV 2026 **chỉ có chức danh**, không có mô tả công việc, nên kỹ năng được đoán từ tiêu đề: "tin có kỹ năng" thấp không có nghĩa là tin không đòi kỹ năng.

## 2. Mức lương và kinh nghiệm – hai nhóm tách riêng
Trang tách **lương chào tuyển 2026** khỏi **thu nhập tháng 2023** (ADR 0007), để không trộn hai đại lượng khác nhau.

In [ ]:
for cohort in ["salary", "income"]:
    analysis = service.salary_analysis(cohort, ("roles", "regions", "skills", "experience"))
    print(f"== {cohort}: {analysis['posting_count']} tin · đại lượng: {analysis['quantity']}")
    display(pd.DataFrame(analysis["roles"]).rename(columns={"name": "Nhóm nghề", "posting_count": "Số tin", "mean_vnd_month": "Trung điểm TB (VND/tháng)"}))

Chỉ 14 tin lương và 2 tin thu nhập: **quá ít để kết luận**. Trang ghi rõ "Hai tin chưa đủ để kết luận xu hướng". Đây chính là lý do dự án mở rộng sang bộ salary-v2 (1.735 tin) cho mô hình và trang Tổng quan.

## Kết luận
- Trang Thị trường minh họa phiên bản "ít nhưng chắc chắn": mọi con số truy được về tin gốc đã kiểm duyệt.
- Bộ lọc biến bức tranh chung thành câu trả lời riêng cho từng nghề, từng nơi.
- Mức lương gắn với kỹ năng chỉ là tương quan, không chứng minh kỹ năng gây tăng lương.